# Phase 5: Regime Detection and Backtesting

This notebook demonstrates the regime change detector, runs full backtests
across all filter variants, and compares performance via Brier score, log loss,
and calibration analysis.

## Outline
1. Regime detection on synthetic data with known change points
2. Full backtest: compare all filter variants
3. Calibration curve analysis
4. Summary comparison table

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from loguru import logger
logger.disable('src')

from src.data.synthetic import generate_random_walk, generate_step_change
from src.filters.scalar_kalman import ScalarKalmanFilter
from src.detection.regime_detector import RegimeDetector
from src.pipeline.backtest import FilterBacktest
from src.analysis.metrics import calibration_curve, brier_score

sns.set_theme(style='whitegrid')
%matplotlib inline

## 1. Regime Detection on Data with Known Change Points

In [ ]:
# Generate data with step change
data = generate_step_change(
    n_steps=500, step_time=250,
    step_from=0.35, step_to=0.72,
    Q=1e-6, R=1e-3, seed=42,
)

# Run filter
kf = ScalarKalmanFilter(Q=1e-5, R=1e-3)
result = kf.filter(data.observations)

# Run regime detector
detector = RegimeDetector(window=20, cusum_threshold=4.0)
alerts = []
for t in range(len(data.observations)):
    alert = detector.check(
        innovation=result.innovations[t],
        S=result.innovation_covariances[t],
    )
    alerts.append(alert)

detection_times = [t for t, a in enumerate(alerts) if a.detected]
print(f'Detections at steps: {detection_times}')
print(f'True change at step 250')

if detection_times:
    first_detect = min(t for t in detection_times if t >= 245)
    print(f'Detection latency: {first_detect - 250} steps')

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Price and filter
ax1.plot(data.observations, color='gray', alpha=0.4, linewidth=0.8, label='Raw')
ax1.plot(data.true_states, color='green', linewidth=1.0, alpha=0.6, label='True')
ax1.plot(result.states, color='#00CED1', linewidth=1.5, label='Filtered')
ax1.axvline(x=250, color='red', linestyle='--', alpha=0.5, label='True change')
for t in detection_times:
    ax1.axvline(x=t, color='orange', alpha=0.3, linewidth=1)
ax1.set_ylabel('Probability')
ax1.set_title('Filtered Price with Regime Detection')
ax1.legend(loc='best', fontsize=9)

# Normalized innovations
S = result.innovation_covariances
normalized = result.innovations / np.sqrt(np.maximum(S, 1e-15))
ax2.plot(normalized, color='#00CED1', alpha=0.7, linewidth=0.8)
ax2.axhline(y=2, color='red', linestyle='--', alpha=0.3)
ax2.axhline(y=-2, color='red', linestyle='--', alpha=0.3)
ax2.axvline(x=250, color='red', linestyle='--', alpha=0.5)
ax2.set_ylabel('Normalized Innovation')
ax2.set_title('Innovation Sequence')

# Detection severity
severities = [a.severity if a.detected else 0 for a in alerts]
ax3.bar(range(len(severities)), severities, color='orange', width=1.0)
ax3.axvline(x=250, color='red', linestyle='--', alpha=0.5, label='True change')
ax3.set_xlabel('Time step')
ax3.set_ylabel('Detection Severity')
ax3.set_title('Regime Change Detections')
ax3.legend()

plt.tight_layout()
plt.show()

## 2. Full Backtest: All Filter Variants

In [ ]:
# Generate multiple synthetic "resolved" markets
scenarios = [
    {'name': 'Resolves YES (high prob)', 'x0': 0.75, 'Q': 1e-5, 'R': 2e-3, 'outcome': 1},
    {'name': 'Resolves NO (low prob)', 'x0': 0.25, 'Q': 1e-5, 'R': 2e-3, 'outcome': 0},
    {'name': 'Resolves YES (volatile)', 'x0': 0.55, 'Q': 5e-4, 'R': 5e-3, 'outcome': 1},
    {'name': 'Resolves NO (volatile)', 'x0': 0.45, 'Q': 5e-4, 'R': 5e-3, 'outcome': 0},
    {'name': 'Resolves YES (low noise)', 'x0': 0.80, 'Q': 1e-6, 'R': 5e-4, 'outcome': 1},
    {'name': 'Resolves NO (uncertain)', 'x0': 0.50, 'Q': 1e-4, 'R': 1e-2, 'outcome': 0},
]

all_results = []
for i, sc in enumerate(scenarios):
    data = generate_random_walk(
        n_steps=300, Q=sc['Q'], R=sc['R'], x0=sc['x0'], seed=42+i,
    )
    bt = FilterBacktest(Q=sc['Q'], R=sc['R'])
    results = bt.run(data.observations, outcome=sc['outcome'])
    for r in results:
        all_results.append({
            'Market': sc['name'],
            'Filter': r.filter_name,
            'Brier Score': r.brier,
            'Log Loss': r.logloss,
            'Innovation %': f'{r.innovation_frac_2sigma:.1%}' if r.innovation_frac_2sigma > 0 else 'N/A',
        })

df = pd.DataFrame(all_results)
print('Backtest Results:')
df_pivot = df.pivot_table(index='Filter', columns='Market', values='Brier Score', aggfunc='mean')
df_pivot

In [ ]:
# Average Brier score by filter type
avg_brier = df.groupby('Filter')['Brier Score'].mean().sort_values()

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#FF6347', '#00CED1', '#9370DB', '#FFD700', '#888888']
bars = ax.barh(avg_brier.index, avg_brier.values, color=colors[:len(avg_brier)])
ax.set_xlabel('Average Brier Score (lower is better)')
ax.set_title('Filter Comparison: Average Brier Score Across All Markets')

for bar, val in zip(bars, avg_brier.values):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=10)

plt.tight_layout()
plt.show()

## 3. Calibration Analysis

In [ ]:
# Aggregate predictions across all markets for calibration curve
# Use the scalar Kalman filter predictions
all_preds = []
all_outcomes = []

for i, sc in enumerate(scenarios):
    data = generate_random_walk(
        n_steps=300, Q=sc['Q'], R=sc['R'], x0=sc['x0'], seed=42+i,
    )
    kf = ScalarKalmanFilter(Q=sc['Q'], R=sc['R'])
    result = kf.filter(data.observations)
    all_preds.extend(result.states)
    all_outcomes.extend([sc['outcome']] * len(result.states))

all_preds = np.array(all_preds)
all_outcomes = np.array(all_outcomes)

bins, actual, counts = calibration_curve(all_preds, all_outcomes, n_bins=10)

fig, ax = plt.subplots(figsize=(8, 8))
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect calibration')
ax.plot(bins, actual, 'o-', color='#00CED1', markersize=8, label='Kalman filter')

# Size points by count
for b, a, c in zip(bins, actual, counts):
    if c > 0:
        ax.annotate(f'n={int(c)}', (b, a), textcoords='offset points',
                    xytext=(5, 5), fontsize=8, alpha=0.7)

ax.set_xlabel('Predicted Probability')
ax.set_ylabel('Observed Frequency')
ax.set_title('Calibration Curve: Scalar Kalman Filter')
ax.legend()
ax.set_xlim(-0.05, 1.05)
ax.set_ylim(-0.05, 1.05)
plt.tight_layout()
plt.show()

## 4. Summary Table

In [ ]:
summary = df.groupby('Filter').agg({
    'Brier Score': ['mean', 'std'],
    'Log Loss': ['mean', 'std'],
}).round(4)

summary.columns = ['Brier Mean', 'Brier Std', 'LogLoss Mean', 'LogLoss Std']
summary = summary.sort_values('Brier Mean')

print('\nFinal Performance Summary:')
print('=' * 65)
print(summary.to_string())
print('\nBest filter by Brier Score:', summary.index[0])